# Summary table of indicators by category.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd


# ============================================================
# Step 1. File names
# ============================================================


INPUT_FILE = Path("combined_country_availability_final_with_indicator_coding.csv")
CLASSIFIED_OUTPUT = Path("step1_indicator_monetary_classification.csv")
SUMMARY_OUTPUT = Path("step1_indicator_category_summary.csv")


# ============================================================
# Step 2. Required columns
# ============================================================

ID_COLUMNS = ["no", "indicator", "Definition", "unit", "source", "original", "dataset"]

SOURCE_CATEGORY_COLUMNS = [
    "Stock",
    "Flow",
    "Policy_Guide",
    "Action",
    "HPS",
    "DDP",
    "MS-NMS",
    "ADE",
]


# ============================================================
# Step 3. Monetary identification patterns
# ============================================================
# This rule identifies monetary-value indicators using text patterns
# in the indicator name, definition, unit, and source fields.

MONETARY_PATTERNS = {
    "currency_usd_dollar": r"\b(?:us\s*dollars?|u\.s\.\s*dollars?|usd|us\$|\$)\b",
    "currency_local": r"\b(?:lcu|local currency|national currency|domestic currency|current lcu|constant lcu)\b",
    "ppp_money": r"\b(?:ppp|purchasing power parity)\b",
    "price_value_cost": r"\b(?:price|prices|priced|value|values|valuation|monetary|market value|net present value)\b",
    "income_wealth_asset_liability": r"\b(?:income|wealth|capital stock|assets?|liabilities?|net worth)\b",
    "fiscal_terms": r"\b(?:expenditure|expenditures|expense|expenses|spending|cost|costs|revenue|revenues|tax|taxes|subsidy|subsidies|tariff|fee|fees)\b",
    "finance_terms": r"\b(?:investment|investments|finance|financing|financial|credit|loan|loans|debt|compensation|wage|wages)\b",
    "gdp_money_denominator": r"\b(?:gdp|gross domestic product|gni|gross national income)\b",
}

NON_MONETARY_EXPLANATION = (
    "Excluded from the retained HPS/DDP/MS-NMS/ADE category because the indicator "
    "does not match the monetary indicator pattern."
)


# ============================================================
# Step 4. Helper functions
# ============================================================

def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).lower()


def monetary_matches(row: pd.Series) -> list[str]:
    searchable_text = " | ".join(
        normalize_text(row.get(col, ""))
        for col in ["indicator", "Definition", "unit", "source"]
    )

    return [
        label
        for label, pattern in MONETARY_PATTERNS.items()
        if re.search(pattern, searchable_text, flags=re.IGNORECASE)
    ]


def add_classification_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    matches = out.apply(monetary_matches, axis=1)
    out["monetary_indicator"] = matches.map(bool).astype(int)
    out["monetary_pattern_matches"] = matches.map(lambda items: "; ".join(items))

    out["Physical stock"] = ((out["Stock"] == 1) & (out["monetary_indicator"] == 0)).astype(int)
    out["Monetary stock"] = ((out["Stock"] == 1) & (out["monetary_indicator"] == 1)).astype(int)
    out["Physical flow"] = ((out["Flow"] == 1) & (out["monetary_indicator"] == 0)).astype(int)
    out["Monetary flow"] = ((out["Flow"] == 1) & (out["monetary_indicator"] == 1)).astype(int)

    out["Action_kept"] = out["Action"].astype(int)
    out["Policy Guide_kept"] = out["Policy_Guide"].astype(int)

    monetary_only_categories = ["HPS", "DDP", "MS-NMS", "ADE"]

    for category in monetary_only_categories:
        out[f"{category}_monetary_only"] = (
            (out[category] == 1) & (out["monetary_indicator"] == 1)
        ).astype(int)

        out[f"{category}_non_monetary_explanation"] = ""

        needs_explanation = (out[category] == 1) & (out["monetary_indicator"] == 0)

        out.loc[needs_explanation, f"{category}_non_monetary_explanation"] = (
            NON_MONETARY_EXPLANATION
        )

    return out


def build_summary(df: pd.DataFrame) -> pd.DataFrame:
    total_indicators = len(df)

    categories = [
        ("Physical stock", "Stock indicators not measured in monetary terms"),
        ("Monetary stock", "Stock indicators matching the monetary indicator pattern"),
        ("Physical flow", "Flow indicators not measured in monetary terms"),
        ("Monetary flow", "Flow indicators matching the monetary indicator pattern"),
        ("Action_kept", "Action indicators kept as coded"),
        ("Policy Guide_kept", "Policy Guide indicators kept as coded"),
        ("HPS_monetary_only", "HPS indicators retained only when monetary"),
        ("DDP_monetary_only", "DDP indicators retained only when monetary"),
        ("MS-NMS_monetary_only", "MS-NMS indicators retained only when monetary"),
        ("ADE_monetary_only", "ADE indicators retained only when monetary"),
    ]

    rows = []

    for column, description in categories:
        count = int(df[column].sum())

        rows.append(
            {
                "category": column,
                "description": description,
                "indicator_count": count,
                "percent_of_all_indicators": round(count / total_indicators * 100, 2),
            }
        )

    rows.append(
        {
            "category": "Any monetary indicator",
            "description": "Indicators matching at least one monetary pattern",
            "indicator_count": int(df["monetary_indicator"].sum()),
            "percent_of_all_indicators": round(
                df["monetary_indicator"].sum() / total_indicators * 100, 2
            ),
        }
    )

    return pd.DataFrame(rows)


# ============================================================
# Step 5. Run classification
# ============================================================

df = pd.read_csv(INPUT_FILE)

missing_columns = [
    col for col in ID_COLUMNS + SOURCE_CATEGORY_COLUMNS
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required input columns: {missing_columns}")

classified = add_classification_columns(df)
summary = build_summary(classified)

classified.to_csv(CLASSIFIED_OUTPUT, index=False, encoding="utf-8-sig")
summary.to_csv(SUMMARY_OUTPUT, index=False, encoding="utf-8-sig")

print("Step 1 monetary classification complete")
print(f"Input rows: {len(df):,}")
print(f"Classified output: {CLASSIFIED_OUTPUT}")
print(f"Summary output: {SUMMARY_OUTPUT}")
print()
print(summary.to_string(index=False))

Step 1 monetary classification complete
Input rows: 1,320
Classified output: step1_indicator_monetary_classification.csv
Summary output: step1_indicator_category_summary.csv

              category                                              description  indicator_count  percent_of_all_indicators
        Physical stock          Stock indicators not measured in monetary terms               52                       3.94
        Monetary stock Stock indicators matching the monetary indicator pattern               91                       6.89
         Physical flow           Flow indicators not measured in monetary terms              202                      15.30
         Monetary flow  Flow indicators matching the monetary indicator pattern              221                      16.74
           Action_kept                          Action indicators kept as coded               14                       1.06
     Policy Guide_kept                    Policy Guide indicators kept as coded  